# День 1. Как обучается нейросеть
### От градиента к экспериментам с батчами и dropout

**Science School AI360 · 3 октября 2026 · студенческая версия**

За 3–4 часа вы соберёте цикл обучения классификатора рукописных цифр и проверите,
как размер батча и dropout влияют на обучение. Общая часть подготовит к двум статьям:
[Keskar et al. — On Large-Batch Training for Deep Learning](https://arxiv.org/abs/1609.04836)
и [Srivastava et al. — Dropout](https://jmlr.org/papers/v15/srivastava14a.html).

Нужны Python, производные, матрицы и математическое ожидание. Опыт в DL не требуется.
Сегодня достаточно CPU; при наличии CUDA-GPU ноутбук использует его автоматически.

К концу занятия вы сможете объяснить, что именно делают `forward`, `backward` и `step`,
обучить MLP, отличить ошибку обучения от ошибки на новых данных и сформулировать
ограничения первых экспериментов с батчами и dropout.


## Маршрут занятия

| Блок | Время | Что получится |
|---|---:|---|
| Подготовка и тензоры · E1 | 20 мин | Аффинный слой и его размерности |
| Градиент и autograd · E2–E3 | 35 мин | Обучение прямой двумя способами |
| Данные и cross-entropy · E4 | 30 мин | Корректная задача классификации |
| Обучение и оценка · E5–E6 | 40 мин | Работающий линейный baseline |
| Перерыв | 10 мин | |
| Нелинейная сеть · E7 | 20 мин | MLP и роль ReLU |
| Батчи и шум градиента · E8 | 30 мин | Кривые обучения и измерение шума |
| Dropout · E9 | 30 мин | Случайная маска и первое сравнение |
| Обсуждение и план следующего дня | 10 мин | Проверяемая гипотеза |

**Всего около 3 ч 45 мин.** Один из заключительных блоков можно перенести на следующий день.

Девять ячеек помечены **E1–E9**. В них замените `raise NotImplementedError(...)` своим
кодом и выполните следующую проверку. Пока пропуски не заполнены, `Run all` закономерно
останавливается. Не меняйте проверки, чтобы добиться зелёного сообщения.

Подсказки раскрываются по нажатию. После исправления функции заново выполните ячейку
с её определением. После изменения данных, модели или настроек заново запустите зависимые
эксперименты. Перед итоговым сравнением полезно выполнить **Restart kernel → Run all**.

Зелёная проверка означает, что пройден набор контрольных примеров. Она не заменяет
объяснение решения и не доказывает корректность для любых возможных входов.


## 0. Подготовка

Откройте `.ipynb` в JupyterLab, VS Code или загрузите его в Google Colab через меню
открытия ноутбука. В Colab при желании выберите среду с GPU. Файл не требует других
локальных Python-модулей. MNIST скачивается при первом обращении и затем читается из кэша.

В учебной среде нужны Python 3.10+ и пакеты `torch`, `torchvision`, `numpy`, `matplotlib`.
Если они уже есть, установка не нужна. Если импорт ниже сообщает об отсутствии пакета,
раскомментируйте установочную строку, выполните её и перезапустите kernel.
Для локальной CUDA-среды установку согласованных версий torch/torchvision лучше выполнить
заранее по [официальному установщику PyTorch](https://pytorch.org/get-started/locally/).


In [ ]:
# Только если отсутствуют зависимости; в готовой Colab-среде обычно не требуется.
# %pip install torch torchvision numpy matplotlib


In [ ]:
import os
import math
import time
import random
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.data import DataLoader, TensorDataset
from torchvision.datasets import MNIST
from IPython import get_ipython

if get_ipython() is not None:
    get_ipython().run_line_magic('matplotlib', 'inline')

plt.rcParams.update({
    'figure.figsize': (8, 4), 'figure.dpi': 110,
    'axes.spines.top': False, 'axes.spines.right': False,
    'axes.grid': True, 'grid.alpha': 0.2,
    'font.family': 'DejaVu Sans',
})

SEED = 360
DATA_DIR = Path(os.environ.get('AI360_DATA_DIR', './data'))
DEVICE_PREFERENCE = os.environ.get('AI360_DEVICE', 'auto')  # либо 'cpu', 'cuda', 'mps'
if DEVICE_PREFERENCE != 'auto':
    device = torch.device(DEVICE_PREFERENCE)
elif torch.cuda.is_available():
    device = torch.device('cuda')
elif hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
    device = torch.device('mps')
else:
    device = torch.device('cpu')

torch.set_num_threads(min(4, os.cpu_count() or 1))

def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    if hasattr(torch.backends, 'cudnn'):
        torch.backends.cudnn.benchmark = False
        torch.backends.cudnn.deterministic = True

seed_everything(SEED)
print('PyTorch:', torch.__version__, '| устройство:', device)
print('Seed:', SEED)
print('Одинаковый seed не гарантирует побитового совпадения на разных устройствах.')


## 1. Нейросеть начинается с матриц

**Тензор** — многомерный массив. Для начала используем те же операции, что с матрицами.
Пусть в батче $B$ объектов, каждый описан $D$ признаками. Тогда $X$ имеет форму $[B,D]$.
Один аффинный слой вычисляет:

$$Z=XW+b,\qquad W\in\mathbb R^{D\times K},\quad b\in\mathbb R^K.$$

Результат имеет форму $[B,K]$. Вектор $b$ прибавляется к каждой строке — это **broadcasting**.
Параметры $W,b$ общие для всех объектов. Батч не означает, что для каждого объекта
создаётся отдельная модель.

**E1 · 5–7 минут.** Реализуйте `affine` в одну строку, без циклов по объектам.
Сначала предскажите форму результата. В PyTorch `@` — матричное произведение,
а `*` — поэлементное умножение.

<details><summary>Подсказка</summary>
Внутренние размерности матричного произведения должны совпадать: $[B,D]@[D,K]$.
Не транспонируйте $W$: здесь выбрана именно форма $[D,K]$.
</details>


In [ ]:
def affine(x, W, b):
    # TODO E1: матричное произведение и добавление смещения.
    raise NotImplementedError('E1: заполните аффинный слой')


In [ ]:
x_check = torch.tensor([[1., 2.], [3., 4.]])
W_check = torch.tensor([[1., 0., -1.], [0., 2., 1.]])
b_check = torch.tensor([0.5, 1., -0.5])
z_check = affine(x_check, W_check, b_check)
assert z_check.shape == (2, 3), 'Ожидается [число объектов, число выходов].'
torch.testing.assert_close(z_check, torch.tensor([[1.5, 5., 0.5], [3.5, 9., 0.5]]))
assert affine(torch.ones(5, 2), W_check, b_check).shape == (5, 3)
print('E1 ✓ — формы и значения верны')


## 2. Обучение — это изменение параметров по градиенту

Начнём с регрессии: по числу $x$ предсказываем $y$ прямой $\hat y=wx+b$.
Предсказание зависит от параметров; **loss** сообщает, насколько оно ошибочно:

$$L(w,b)=\frac1n\sum_i(wx_i+b-y_i)^2.$$

Градиент показывает направление наиболее быстрого локального **увеличения** loss.
Поэтому градиентный спуск делает шаг с противоположным знаком:

$$w\leftarrow w-\eta\frac{\partial L}{\partial w},\qquad
b\leftarrow b-\eta\frac{\partial L}{\partial b}.$$

Learning rate $\eta$ — длина шага с учётом масштаба градиента. Слишком маленький шаг
замедляет обучение, слишком большой способен сделать его неустойчивым.

**E2 · 8 минут.** На бумаге выведите обе производные, затем реализуйте функцию,
возвращающую три скалярных тензора: loss, $\partial L/\partial w$, $\partial L/\partial b$.
В этой ячейке пока не используйте autograd.

<details><summary>Подсказка</summary>
Обозначьте ошибку одного объекта через $e_i=wx_i+b-y_i$. Примените правило цепочки
к $e_i^2$ и только потом усредните по объектам. Проверьте множитель 2.
</details>


In [ ]:
def mse_and_grad(x, y, w, b):
    # x, y: одномерные тензоры одной длины; w, b: скалярные тензоры.
    # TODO E2: вычислите предсказания, среднюю квадратичную ошибку и две производные.
    raise NotImplementedError('E2: верните loss, dw, db')


In [ ]:
xc = torch.tensor([0., 1., 2.], dtype=torch.float64)
yc = torch.tensor([1., 3., 5.], dtype=torch.float64)
wc, bc = torch.tensor(0., dtype=torch.float64), torch.tensor(0., dtype=torch.float64)
lc, dwc, dbc = mse_and_grad(xc, yc, wc, bc)
torch.testing.assert_close(lc, torch.tensor(35/3, dtype=torch.float64))
torch.testing.assert_close(dwc, torch.tensor(-26/3, dtype=torch.float64))
torch.testing.assert_close(dbc, torch.tensor(-6., dtype=torch.float64))
# Независимая численная проверка производной в другой точке.
eps = 1e-5
w0, b0 = torch.tensor(0.7, dtype=torch.float64), torch.tensor(-0.2, dtype=torch.float64)
finite_dw = ((((w0+eps)*xc+b0-yc)**2).mean() - (((w0-eps)*xc+b0-yc)**2).mean())/(2*eps)
_, analytic_dw, _ = mse_and_grad(xc, yc, w0, b0)
torch.testing.assert_close(analytic_dw, finite_dw)
print('E2 ✓ — ручные производные согласуются с конечной разностью')


In [ ]:
g = torch.Generator().manual_seed(SEED)
x_reg = torch.linspace(-1, 1, 120)
y_reg = 2.0 * x_reg + 0.5 + 0.15 * torch.randn(120, generator=g)
w_manual, b_manual = torch.tensor(0.), torch.tensor(0.)
manual_history = []
for _ in range(80):
    loss, dw, db = mse_and_grad(x_reg, y_reg, w_manual, b_manual)
    manual_history.append(loss.item())
    w_manual -= 0.1 * dw
    b_manual -= 0.1 * db

fig, axes = plt.subplots(1, 2, figsize=(11, 3.7))
axes[0].scatter(x_reg, y_reg, s=12, alpha=0.55, label='Наблюдения')
axes[0].plot(x_reg, w_manual*x_reg+b_manual, color='#D75D3A', label='Обученная прямая')
axes[0].set(xlabel='x', ylabel='y', title='Подбор параметров')
axes[0].legend()
axes[1].plot(manual_history)
axes[1].set(xlabel='Шаг', ylabel='MSE', title='Ошибка во время обучения')
fig.tight_layout()
plt.show()
print(f'w = {w_manual.item():.3f}, b = {b_manual.item():.3f}')


### Зачем нужен backpropagation

В сети с миллионами параметров неудобно выписывать каждую производную вручную.
**Backpropagation** применяет правило цепочки от скалярного loss назад по вычислениям.
PyTorch строит граф операций над тензорами с `requires_grad=True`; вызов `loss.backward()`
записывает производные в `.grad`. Это аналитическое дифференцирование вычислительного
графа, а не перебор параметров конечными разностями.

**E3 · 10 минут.** Дополните один шаг обучения. Посчитайте loss, вызовите обратный проход,
измените оба параметра **на месте** внутри `torch.no_grad()` и очистите `.grad` после шага.
Верните числовое значение loss до обновления.

Почему два дополнительных действия? Обновление весов сейчас не должно входить в граф
следующего шага, а PyTorch по умолчанию **накапливает** градиенты между обратными проходами.

<details><summary>Подсказка</summary>
Нужны `.backward()`, поля `w.grad` и `b.grad`, контекст `with torch.no_grad():`.
Для очистки можно присвоить полям `.grad` значение `None`.
Не создавайте новые локальные тензоры вместо обновления переданных параметров.
</details>


In [ ]:
def autograd_step(x, y, w, b, lr):
    # TODO E3: loss -> backward -> обновление w,b без записи в граф -> очистка .grad.
    # Верните float — значение loss до обновления параметров.
    raise NotImplementedError('E3: реализуйте шаг с autograd')


In [ ]:
xt = torch.tensor([-1., 0., 1.])
yt = 2 * xt + 0.5
wt, bt = torch.tensor(0., requires_grad=True), torch.tensor(0., requires_grad=True)
reported = autograd_step(xt, yt, wt, bt, 0.1)
assert isinstance(reported, float)
torch.testing.assert_close(wt.detach(), torch.tensor(0.26666667))
torch.testing.assert_close(bt.detach(), torch.tensor(0.1))
assert wt.is_leaf and bt.is_leaf and wt.requires_grad and bt.requires_grad
assert wt.grad is None or torch.count_nonzero(wt.grad) == 0
assert bt.grad is None or torch.count_nonzero(bt.grad) == 0
for _ in range(150):
    autograd_step(xt, yt, wt, bt, 0.1)
torch.testing.assert_close(wt.detach(), torch.tensor(2.), atol=1e-4, rtol=1e-4)
torch.testing.assert_close(bt.detach(), torch.tensor(0.5), atol=1e-4, rtol=1e-4)
print('E3 ✓ — параметры обновляются, граф и градиенты не накапливаются')


## 3. Задача классификации и честное разбиение данных

MNIST — изображения цифр размером $28\times28$, метки — целые числа от 0 до 9.
Входы превратим в числа от 0 до 1, разделив пиксели на 255. Это фиксированное
преобразование, для него не нужно оценивать статистики на данных.

**Train** используется для изменения весов. **Validation** — для сравнения моделей
и выбора настроек. **Test** оставляют для итоговой независимой оценки, когда решения
уже приняты. Сегодня официальный test MNIST не читаем и не оцениваем.

Из официальной обучающей части выделим непересекающиеся подмножества по классам:
6000 объектов для обучения и 2000 для validation. Это учебный масштаб, а не точное
воспроизведение чисел статьи. Все сравниваемые модели увидят одни и те же объекты.
Остальные примеры сегодня не используем.


In [ ]:
try:
    mnist = MNIST(root=DATA_DIR, train=True, download=True)
except Exception as exc:
    raise RuntimeError(
        'Не удалось загрузить MNIST. Проверьте интернет или заранее скопируйте '
        'кэш MNIST в DATA_DIR. Затем повторите эту ячейку.'
    ) from exc

TRAIN_PER_CLASS = 600
VAL_PER_CLASS = 200
split_rng = torch.Generator().manual_seed(SEED)
train_parts, val_parts = [], []
for label in range(10):
    indices = torch.where(mnist.targets == label)[0]
    indices = indices[torch.randperm(len(indices), generator=split_rng)]
    val_parts.append(indices[:VAL_PER_CLASS])
    train_parts.append(indices[VAL_PER_CLASS:VAL_PER_CLASS + TRAIN_PER_CLASS])
train_indices = torch.cat(train_parts)
val_indices = torch.cat(val_parts)
assert not set(train_indices.tolist()) & set(val_indices.tolist())

def make_dataset(indices):
    images = mnist.data[indices].unsqueeze(1).float() / 255.0
    labels = mnist.targets[indices].long()
    return TensorDataset(images, labels)

train_ds = make_dataset(train_indices)
val_ds = make_dataset(val_indices)
print('Train:', len(train_ds), '| validation:', len(val_ds))
print('Один объект:', train_ds[0][0].shape, '| метка:', train_ds[0][1].item())
print('Классы train:', torch.bincount(train_ds.tensors[1]).tolist())
print('Классы validation:', torch.bincount(val_ds.tensors[1]).tolist())


In [ ]:
preview_rng = torch.Generator().manual_seed(SEED + 1)
preview_ids = torch.randperm(len(train_ds), generator=preview_rng)[:12]
fig, axes = plt.subplots(2, 6, figsize=(10, 3.7))
for ax, idx in zip(axes.flat, preview_ids):
    image, label = train_ds[int(idx)]
    ax.imshow(image.squeeze(0), cmap='gray', vmin=0, vmax=1)
    ax.set_title(f'Цифра {label.item()}')
    ax.axis('off')
fig.tight_layout()
plt.show()


### Из десяти чисел — в вероятности и loss

Классификатор выдаёт для каждого объекта десять **логитов** — произвольных вещественных
оценок классов. Softmax превращает их в вероятности:

$$p_k=\frac{e^{z_k}}{\sum_j e^{z_j}}.$$

Правильной метке $y$ соответствует cross-entropy $-\log p_y$. Для батча усредняем её:

$$L=\frac1B\sum_i\left(\log\sum_k e^{z_{ik}}-z_{i,y_i}\right).$$

Если все 10 классов равновероятны, loss равен $\log 10\approx2.303$.
**Accuracy** считает долю правильных классов. Она не показывает степень уверенности;
cross-entropy отличает осторожную ошибку от уверенной. Accuracy с `argmax` не подходит
в качестве дифференцируемого loss для обычного градиентного обучения.

**E4 · 8 минут.** Реализуйте среднюю cross-entropy из сырых логитов через
`torch.logsumexp` или `torch.log_softmax`, без `F.cross_entropy` в самой функции.
Верните скалярный тензор, сохранив возможность дифференцирования.

<details><summary>Подсказка</summary>
Строки — объекты, столбцы — классы. Нужный логит выбирается по индексу метки
в каждой строке. Суммировать экспоненты надо по классам, а loss усреднять по объектам.
Наивное `exp` для больших логитов переполняется.
</details>


In [ ]:
def cross_entropy_from_logits(logits, targets):
    # logits: [B, K], targets: [B] с целыми индексами классов.
    # TODO E4: стабильная средняя cross-entropy, без ручного вычисления exp.
    raise NotImplementedError('E4: вычислите cross-entropy')


In [ ]:
zc = torch.tensor([[1., -1., 2.], [-2., 1., 0.]], dtype=torch.float64, requires_grad=True)
tc = torch.tensor([2, 0])
custom_loss = cross_entropy_from_logits(zc, tc)
torch.testing.assert_close(custom_loss, F.cross_entropy(zc, tc))
torch.testing.assert_close(cross_entropy_from_logits(zc + 1000., tc), custom_loss)
uniform_loss = cross_entropy_from_logits(torch.zeros(4, 10), torch.tensor([1, 3, 5, 8]))
torch.testing.assert_close(uniform_loss, torch.tensor(math.log(10)))
assert torch.isfinite(torch.autograd.grad(custom_loss, zc)[0]).all()
print('E4 ✓ — loss устойчив к большим логитам и поддерживает градиент')


Дальше используем готовую `F.cross_entropy`: свою реализацию уже проверили.
Она получает **сырые логиты**, поэтому перед ней не нужен `softmax`.
Метки в нашей постановке имеют форму `[B]` и тип `torch.long`.
Вероятности понадобятся отдельно для визуализации предсказаний.


## 4. Собираем обучение классификатора

`nn.Module` хранит параметры модели. В `nn.Sequential` выход одного слоя передаётся
следующему. `nn.Flatten()` превращает каждый объект `[1,28,28]` в вектор длины 784,
сохраняя первую ось батча. `nn.Linear(784,10)` содержит обучаемые веса и смещения.
Его веса хранятся транспонированно относительно соглашения в E1 — это деталь интерфейса.

Линейный классификатор ниже — наш **baseline**: простая модель, с которой надо сравнить
более сложную. Он выдаёт логиты; обучение ещё не происходило.


In [ ]:
def make_linear():
    return nn.Sequential(nn.Flatten(), nn.Linear(28 * 28, 10))

linear_example = make_linear()
print(linear_example)
print('Форма выхода:', linear_example(torch.zeros(3, 1, 28, 28)).shape)
print('Обучаемых параметров:', sum(p.numel() for p in linear_example.parameters()))


**Батч, шаг и эпоха — разные единицы.**

- Батч — несколько объектов, по которым вычисляют один градиент.
- Шаг оптимизатора — одно обновление параметров.
- Эпоха — один проход по обучающей выборке.

При $N$ объектах и размере батча $B$ в нашей реализации будет $\lceil N/B\rceil$
шагов за эпоху: последний неполный батч сохраняется. `DataLoader` нарезает данные
на батчи и, если попросить, перемешивает объекты.

**E5 · 10 минут.** Реализуйте один шаг. Входы уже лежат на нужном устройстве.
Переведите модель в режим обучения, очистите старые градиенты, получите логиты,
вычислите loss, сделайте backward и обновите параметры оптимизатором. Верните `float` loss.

Дальше мы используем Adam с фиксированным learning rate. Это оптимизатор, который
адаптирует обновления по истории градиентов; самостоятельно реализовывать его сегодня
не нужно. Backward вычисляет производные, а `optimizer.step()` меняет параметры.

<details><summary>Подсказка</summary>
Полезные методы: `model.train()`, `optimizer.zero_grad()`, `loss.backward()`,
`optimizer.step()`. Они выполняют разные действия. Не применяйте softmax перед loss.
</details>


In [ ]:
def train_step(model, x, y, optimizer):
    # TODO E5: режим обучения -> очистка -> forward -> loss -> backward -> step.
    raise NotImplementedError('E5: заполните один шаг обучения')


In [ ]:
seed_everything(11)
toy_x = torch.tensor([[-1., -1.], [-1., 0.], [1., 0.], [1., 1.]])
toy_y = torch.tensor([0, 0, 1, 1])
toy_model = nn.Linear(2, 2)
toy_optimizer = torch.optim.SGD(toy_model.parameters(), lr=0.2)
before_loss = F.cross_entropy(toy_model(toy_x), toy_y).item()
toy_model.eval()
for _ in range(40):
    step_loss = train_step(toy_model, toy_x, toy_y, toy_optimizer)
assert isinstance(step_loss, float) and math.isfinite(step_loss)
assert toy_model.training, 'Шаг обучения должен включать режим train.'
assert F.cross_entropy(toy_model(toy_x), toy_y).item() < before_loss * 0.4
# При lr=0 веса не меняются: два одинаковых шага должны дать одинаковые .grad.
probe_model = nn.Linear(2, 2)
probe_optimizer = torch.optim.SGD(probe_model.parameters(), lr=0.0)
train_step(probe_model, toy_x, toy_y, probe_optimizer)
first_grads = [p.grad.clone() for p in probe_model.parameters()]
train_step(probe_model, toy_x, toy_y, probe_optimizer)
for p, expected in zip(probe_model.parameters(), first_grads):
    torch.testing.assert_close(p.grad, expected, msg='Проверьте очистку градиентов перед backward.')
print('E5 ✓ — loss уменьшается, градиенты между шагами не накапливаются')


### Оценка качества не должна обучать модель

`model.eval()` переключает поведение слоёв вроде dropout, но **не отключает** вычисление
градиентов. `torch.no_grad()` отключает запись вычислений в граф, но **не переключает**
режим слоёв. При оценке обычно нужны оба. В нашей текущей линейной модели режимы дают
одинаковый forward; чуть позже появится слой, для которого разница существенна.

**E6 · 12 минут.** Реализуйте оценку на всех батчах. Верните словарь
`{'loss': средний_loss, 'accuracy': доля_правильных}`.

Для loss суммируйте потери отдельных объектов, а в конце делите на общее число объектов.
Если усреднить средние потери батчей без учёта их размеров, неполный последний батч
получит слишком большой вес. Accuracy в коде находится в диапазоне от 0 до 1.

<details><summary>Подсказка</summary>
Логиты имеют форму `[B,10]`; класс выбирается через `argmax` по оси классов.
`F.cross_entropy(..., reduction='sum')` даёт сумму потерь для батча.
Не вызывайте ни backward, ни optimizer.step.
</details>


In [ ]:
def evaluate(model, loader, device):
    # TODO E6: eval, no_grad, перенос x,y на device, суммирование loss и верных ответов.
    # Верните {'loss': ..., 'accuracy': ...} с числами типа float.
    raise NotImplementedError('E6: реализуйте оценку без обучения')


In [ ]:
class EvaluationProbe(nn.Module):
    def forward(self, x):
        assert not self.training, 'Не включён eval().'
        assert not torch.is_grad_enabled(), 'Не включён no_grad().'
        return x

eval_logits = torch.tensor([[3., 0.], [0., 2.], [0., 4.], [2., 0.], [0., 5.]])
eval_targets = torch.tensor([0, 1, 0, 0, 0])
eval_loader = DataLoader(TensorDataset(eval_logits, eval_targets), batch_size=2)
eval_probe = EvaluationProbe()
result_check = evaluate(eval_probe, eval_loader, torch.device('cpu'))
assert abs(result_check['accuracy'] - 0.6) < 1e-7
expected_loss = F.cross_entropy(eval_logits, eval_targets).item()
assert abs(result_check['loss'] - expected_loss) < 1e-6, 'Учтён ли размер последнего батча?'
print('E6 ✓ — режим оценки, отсутствие графа и веса батчей проверены')


### Готовый код для запуска экспериментов

Ниже служебные функции: их не нужно переписывать. Найдите в `fit` вызовы ваших
`train_step` и `evaluate`. Все метрики train/validation снимаются **после эпохи**
на фиксированных весах и в одинаковом режиме оценки.

Для каждой модели заново создаются оптимизатор и загрузчик. Перемешивание данных имеет
собственный генератор случайных чисел: позже маски dropout не смогут изменить порядок
объектов. Один и тот же seed загрузчика задаёт одинаковую перестановку на каждой эпохе
для сопоставляемых запусков; различаться может её разбиение на батчи.

Время включает обучение и оценку после каждой эпохи. Оно зависит от устройства,
поэтому не служит точным бенчмарком производительности.


In [ ]:
def training_loader(batch_size, seed):
    return DataLoader(
        train_ds, batch_size=batch_size, shuffle=True, num_workers=0,
        generator=torch.Generator().manual_seed(seed), drop_last=False,
    )

def synchronize_device():
    if device.type == 'cuda':
        torch.cuda.synchronize()
    elif device.type == 'mps':
        torch.mps.synchronize()

def fit(model, batch_size=128, epochs=8, lr=1e-3, seed=SEED):
    seed_everything(seed)
    model = model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    loader = training_loader(batch_size, seed)
    train_eval_loader = DataLoader(train_ds, batch_size=512, shuffle=False, num_workers=0)
    val_loader = DataLoader(val_ds, batch_size=512, shuffle=False, num_workers=0)
    history, steps = [], 0
    synchronize_device()
    started = time.perf_counter()
    for epoch in range(1, epochs + 1):
        for x, y in loader:
            train_step(model, x.to(device), y.to(device), optimizer)
            steps += 1
        train_metrics = evaluate(model, train_eval_loader, device)
        val_metrics = evaluate(model, val_loader, device)
        history.append({
            'epoch': epoch, 'steps': steps,
            'train_loss': train_metrics['loss'], 'val_loss': val_metrics['loss'],
            'train_accuracy': train_metrics['accuracy'], 'val_accuracy': val_metrics['accuracy'],
        })
        if epoch == 1 or epoch == epochs or epoch % 4 == 0:
            print(f"Эпоха {epoch:2d}/{epochs} | шагов {steps:4d} | "
                  f"train loss {train_metrics['loss']:.3f} | "
                  f"val accuracy {val_metrics['accuracy']:.1%}")
    synchronize_device()
    return {'model': model, 'history': history, 'batch_size': batch_size,
            'epochs': epochs, 'lr': lr, 'seed': seed,
            'seconds': time.perf_counter() - started}

def plot_learning(results, x_axis='epoch'):
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    colors = ['#276AA6', '#D75D3A', '#278574']
    for color, (name, run) in zip(colors, results.items()):
        history = run['history']
        xs = [r[x_axis] for r in history]
        for split, style in [('train', '--'), ('val', '-')]:
            axes[0].plot(xs, [r[f'{split}_loss'] for r in history], style,
                         color=color, label=f'{name} · {split}')
            axes[1].plot(xs, [100*r[f'{split}_accuracy'] for r in history], style,
                         color=color, label=f'{name} · {split}')
    x_label = 'Эпоха' if x_axis == 'epoch' else 'Число обновлений весов'
    axes[0].set(xlabel=x_label, ylabel='Cross-entropy', title='Потери')
    axes[1].set(xlabel=x_label, ylabel='Accuracy, %', title='Качество классификации')
    axes[1].legend(fontsize=8)
    fig.tight_layout()
    plt.show()

def print_results(results):
    print(f"{'Модель':<23} {'Train acc':>10} {'Val acc':>10} {'Val loss':>10} {'Шаги':>7} {'Секунды':>9}")
    for name, run in results.items():
        last = run['history'][-1]
        print(f"{name:<23} {last['train_accuracy']:>10.2%} {last['val_accuracy']:>10.2%} "
              f"{last['val_loss']:>10.3f} {last['steps']:>7} {run['seconds']:>9.1f}")


In [ ]:
seed_everything(SEED)
linear_run = fit(make_linear(), batch_size=128, epochs=4, lr=1e-3)
plot_learning({'Линейная модель': linear_run})


**Обсудите вдвоём, прежде чем продолжать.**

1. Почему уменьшение train loss ещё не гарантирует роста validation accuracy?
2. Что будет, если убрать `optimizer.step()`? А если убрать `zero_grad()`?
3. Сколько шагов было в последнем запуске? Посчитайте по размеру train и батча.

**Ваши ответы:** …


## 5. Добавляем нелинейность

Несколько аффинных слоёв подряд без нелинейности всё ещё задают аффинную функцию:
$$(XW_1+b_1)W_2+b_2=X(W_1W_2)+(b_1W_2+b_2).$$

Поэтому между слоями добавим $\operatorname{ReLU}(z)=\max(0,z)$ поэлементно.
**MLP** — многослойная полносвязная сеть: она сама учит промежуточные признаки.
На сегодня достаточно архитектуры:

$$[B,1,28,28]\to[B,784]\to[B,256]\to[B,256]\to[B,10].$$

После каждого скрытого аффинного слоя — ReLU. Последний слой выдаёт логиты,
поэтому после него не ставим ни ReLU, ни softmax. Сеть без BatchNorm и dropout будет
общей исходной точкой для обеих статей.

**E7 · 8 минут.** Соберите `nn.Sequential`: Flatten, Linear, ReLU, Linear, ReLU, Linear.
Размеры указаны выше. Подсчитайте параметры на бумаге: для каждого Linear это
$D_{in}D_{out}+D_{out}$.

<details><summary>Подсказка</summary>
Порядок аргументов `nn.Linear` — число входных и выходных признаков.
`nn.ReLU()` не содержит обучаемых параметров. Не зашивайте размер батча в архитектуру.
</details>


In [ ]:
def make_mlp():
    # TODO E7: верните nn.Sequential для сети 784 -> 256 -> 256 -> 10 с двумя ReLU.
    raise NotImplementedError('E7: соберите MLP')


In [ ]:
mlp_check = make_mlp()
assert isinstance(mlp_check, nn.Sequential), 'В этом задании используем nn.Sequential.'
assert mlp_check(torch.zeros(3, 1, 28, 28)).shape == (3, 10)
assert mlp_check(torch.zeros(1, 1, 28, 28)).shape == (1, 10)
assert sum(isinstance(m, nn.ReLU) for m in mlp_check.modules()) == 2
assert sum(p.numel() for p in mlp_check.parameters()) == 269322
assert isinstance(list(mlp_check.children())[-1], nn.Linear), 'На выходе нужны логиты.'
probe_loss = F.cross_entropy(mlp_check(torch.randn(4, 1, 28, 28)), torch.tensor([1, 2, 3, 4]))
probe_loss.backward()
assert all(p.grad is not None and torch.isfinite(p.grad).all() for p in mlp_check.parameters())
print('E7 ✓ — архитектура, число параметров и обратный проход проверены')


In [ ]:
EPOCHS = 10
BASE_BATCH = 128
LEARNING_RATE = 1e-3

seed_everything(SEED)
initial_mlp = make_mlp()
# Копии начальных параметров останутся на CPU и не изменятся при обучении.
initial_state = {key: value.detach().cpu().clone() for key, value in initial_mlp.state_dict().items()}
mlp_run = fit(initial_mlp, batch_size=BASE_BATCH, epochs=EPOCHS, lr=LEARNING_RATE)
plot_learning({'MLP': mlp_run})
print_results({'Линейная · 4 эпохи': linear_run, 'MLP · 10 эпох': mlp_run})
print('Бюджеты линейной модели и MLP различаются: это учебная демонстрация, не строгая абляция архитектуры.')


### Что значит «обобщает»

Модель обобщает, когда хорошо предсказывает объекты, которые не участвовали в обновлении
весов. **Переобучение** — ситуация, в которой подгонка train не сопровождается
соответствующим улучшением на новых данных; при продолжении обучения validation loss
может даже расти. Сам по себе ненулевой разрыв train/validation ещё не доказывает
вредного переобучения: смотрите динамику и альтернативные модели.

**Запишите наблюдение:** на какой эпохе validation loss был минимален? Совпадает ли она
с максимальной accuracy? Видите ли вы признаки переобучения или только разницу между выборками?

**Ваше наблюдение:** …

В дальнейших учебных таблицах сравниваем **последнюю эпоху**, без выбора лучшего checkpoint.
В серьёзном эксперименте правило выбора checkpoint надо определить заранее и применять ко всем методам.


In [ ]:
model_for_errors = mlp_run['model']
model_for_errors.eval()
with torch.no_grad():
    # Только validation, официальный test по-прежнему не используется.
    vx, vy = val_ds.tensors
    pieces = [model_for_errors(vx[i:i+256].to(device)).cpu() for i in range(0, len(vx), 256)]
    val_logits = torch.cat(pieces)
    val_predictions = val_logits.argmax(dim=1)
    val_probabilities = val_logits.softmax(dim=1)
error_ids = torch.where(val_predictions != vy)[0][:8]
if len(error_ids) == 0:
    print('На этой validation-выборке ошибок не найдено.')
else:
    fig, axes = plt.subplots(1, len(error_ids), figsize=(12, 2.4), squeeze=False)
    for ax, idx in zip(axes.flat, error_ids):
        i = int(idx)
        pred = val_predictions[i].item()
        ax.imshow(vx[i].squeeze(0), cmap='gray', vmin=0, vmax=1)
        ax.set_title(f"Ответ: {vy[i].item()}\nМодель: {pred}\np={val_probabilities[i,pred]:.2f}", fontsize=9)
        ax.axis('off')
    fig.tight_layout()
    plt.show()


## 6A. Батчи и шум градиента

Этот блок независим от 6B. Нужны выполненные разделы 0–5.

Градиент по батчу — среднее градиентов его объектов:
$$g_B(\theta)=\frac1B\sum_{i\in\mathcal B}\nabla_\theta\ell_i(\theta).$$

При равномерном случайном выборе батча это оценка градиента всей выборки.
В модели независимых выборок с возвращением $\operatorname{Cov}(g_B)=\Sigma/B$.
Значит, увеличение батча обычно уменьшает шум оценки при фиксированных весах.
Но меняет ли это обобщение? Для ответа нужны отдельные эксперименты с обучением.

**Перед запуском запишите прогноз:** какая модель быстрее снизит train loss по эпохам?
А по шагам? Какое качество на validation ожидаете и насколько уверены?

**Прогноз:** …

Сравним уже обученную MLP с батчем 128 и новую MLP с батчем 1024.
Начальные веса, данные, перестановки по эпохам, learning rate и число эпох одинаковы.
Число обновлений **разное**. Это намеренно простой пилот, в котором есть смешивающий фактор.


In [ ]:
LARGE_BATCH = 1024
large_model = make_mlp()
large_model.load_state_dict(initial_state)
large_run = fit(large_model, batch_size=LARGE_BATCH, epochs=EPOCHS, lr=LEARNING_RATE)
batch_results = {f'Батч {BASE_BATCH}': mlp_run, f'Батч {LARGE_BATCH}': large_run}
plot_learning(batch_results, x_axis='epoch')
plot_learning(batch_results, x_axis='steps')
print_results(batch_results)
for batch in [BASE_BATCH, LARGE_BATCH]:
    print(f'B={batch}: {math.ceil(len(train_ds)/batch)} шагов за эпоху, '
          f'{EPOCHS * math.ceil(len(train_ds)/batch)} за весь запуск')


**Заполните до чтения подсказки.**

- Что произошло с train loss и validation accuracy при увеличении батча? …
- Какая переменная, кроме шума, изменилась вместе с батчем? …
- Какой следующий эксперимент отделил бы эти объяснения? …

<details><summary>Подсказка для обсуждения</summary>
Сравните количество обновлений. При фиксированных эпохах большой батч делает меньше шагов.
Для контроля можно сравнить одинаковое число шагов, но тогда различится число обработанных
примеров. Подбор learning rate — ещё один отдельный контроль. Единственного способа
«уравнять всё» нет: сначала надо сформулировать вопрос.
</details>

### E8 · 8 минут. Измеряем градиент, не обучая модель

Реализуйте `batch_gradient`: получите градиент **средней** cross-entropy по всем
параметрам модели, превратите градиенты в одномерные векторы и склейте их.
Возвращаемый тензор должен быть отсоединён от графа. Веса и существующие `.grad`
изменяться не должны. Используйте `torch.autograd.grad`, без оптимизатора.

<details><summary>Подсказка</summary>
`torch.autograd.grad(loss, tuple(model.parameters()))` возвращает градиенты как значения,
не записывая их в `.grad`. Нужны также `torch.cat`, `.reshape(-1)` и `.detach()`.
Во время измерения модель должна быть детерминированной: используйте `eval()`.
</details>


In [ ]:
def batch_gradient(model, x, y):
    # TODO E8: градиент средней cross-entropy, объединённый в один вектор.
    # Не изменяйте веса, не вызывайте step и не перезаписывайте .grad.
    raise NotImplementedError('E8: вычислите градиент батча без обновления весов')


In [ ]:
seed_everything(9)
gradient_model = nn.Linear(2, 2)
gx = torch.tensor([[1., 2.], [-1., 0.], [0., 1.]])
gy = torch.tensor([0, 1, 0])
weights_before = [p.detach().clone() for p in gradient_model.parameters()]
for parameter in gradient_model.parameters():
    parameter.grad = torch.full_like(parameter, 7.)
g1 = batch_gradient(gradient_model, gx, gy)
g2 = batch_gradient(gradient_model, gx.repeat(2, 1), gy.repeat(2))
assert g1.ndim == 1 and g1.numel() == sum(p.numel() for p in gradient_model.parameters())
assert not g1.requires_grad and torch.isfinite(g1).all()
torch.testing.assert_close(g1, g2, msg='Дублирование батча не меняет градиент среднего loss.')
# Средний градиент группы равен среднему индивидуальных градиентов.
individual = [batch_gradient(gradient_model, gx[i:i+1], gy[i:i+1]) for i in range(len(gy))]
torch.testing.assert_close(g1, torch.stack(individual).mean(dim=0))
for parameter, original in zip(gradient_model.parameters(), weights_before):
    torch.testing.assert_close(parameter.detach(), original)
    torch.testing.assert_close(parameter.grad, torch.full_like(parameter, 7.))
print('E8 ✓ — градиент усреднён правильно, веса и .grad сохранены')


In [ ]:
noise_model = make_mlp().to(device)
noise_model.load_state_dict(initial_state)
noise_rng = torch.Generator().manual_seed(SEED + 99)
noise_ids = torch.randperm(len(train_ds), generator=noise_rng)[:1024]
noise_x = train_ds.tensors[0][noise_ids].to(device)
noise_y = train_ds.tensors[1][noise_ids].to(device)
reference_gradient = batch_gradient(noise_model, noise_x, noise_y)

noise_batches = [8, 32, 128]
TRIALS = 20
noise_means, noise_se = [], []
for batch in noise_batches:
    deviations = []
    for _ in range(TRIALS):
        # Без возвращения внутри батча; новый случайный батч в каждом повторении.
        indices = torch.randperm(len(noise_ids), generator=noise_rng)[:batch].to(device)
        gradient = batch_gradient(noise_model, noise_x[indices], noise_y[indices])
        deviations.append((gradient - reference_gradient).square().sum().item())
    noise_means.append(float(np.mean(deviations)))
    noise_se.append(float(np.std(deviations, ddof=1) / np.sqrt(TRIALS)))

N_ref = len(noise_ids)
shape = np.array([(N_ref-b)/(b*(N_ref-1)) for b in noise_batches])
guide = noise_means[0] * shape / shape[0]
fig, ax = plt.subplots(figsize=(8, 4))
ax.errorbar(noise_batches, noise_means, yerr=noise_se, fmt='o-', capsize=4,
            label='Средний квадрат отклонения градиента')
ax.plot(noise_batches, guide, '--', color='#D75D3A', label='Форма зависимости с поправкой на конечную выборку')
ax.set(xscale='log', yscale='log', xlabel='Размер батча B',
       ylabel='Среднее ||g_B − g_ref||²', title='Шум при одних и тех же весах')
ax.legend(fontsize=8)
fig.tight_layout()
plt.show()
for batch, mean in zip(noise_batches, noise_means):
    print(f'B={batch:3d}: среднее квадратное отклонение = {mean:.6f}')


Здесь $g_{ref}$ — точный средний градиент **подвыборки из 1024 объектов**, а не всего MNIST.
Внутри неё батчи выбираются без возвращения. Тогда зависимость дисперсии от $B$
пропорциональна $(N-B)/(B(N-1))$; при $B\ll N$ она близка к $1/B$.
Пунктир масштабирован по первой экспериментальной точке: это проверка формы зависимости,
а не независимое предсказание её абсолютной величины. Полоски — стандартная ошибка
оценки среднего по случайным батчам; это **не** разброс качества между обучениями сети.

**Связь со статьёй Keskar.** Авторы связывают режим батчей с разными свойствами найденных
решений. Сегодня мы проверили лишь два элемента: шум оценки градиента и поведение обучения.
Мы ещё **не измеряли остроту**, не доказали причинную связь и не воспроизвели полный протокол статьи.
На второй день можно добавить контроль числа шагов, настройку learning rate и срезы loss
в пространстве весов. [Оригинальная статья](https://arxiv.org/pdf/1609.04836).


## 6B. Dropout и регуляризация

Этот блок независим от 6A. Если вы его пропустили, продолжайте здесь после раздела 5.

**Регуляризация** меняет условия обучения так, чтобы ограничить нежелательную подгонку
под train. Dropout во время обучения случайно обнуляет отдельные активации.
Здесь $q$ — вероятность **отключения**, $m_j\sim\operatorname{Bernoulli}(1-q)$:

$$\widetilde h_j=\frac{m_jh_j}{1-q},\qquad 0\leq q<1.$$

Это inverted dropout. Множитель сохраняет математическое ожидание каждой активации:
$\mathbb E[\widetilde h_j]=h_j$. При обычной оценке используем все активации без
маски и без масштабирования. Сохранение ожидания активаций не означает точного
усреднения выхода всей нелинейной сети.

В статье Srivastava $p$ — вероятность **сохранения** нейрона; в `nn.Dropout(p=...)`
параметр означает вероятность отключения. Для ясности в своих функциях называем её $q$.

**E9 · 8 минут.** Реализуйте функцию для произвольного вещественного тензора.
В режиме оценки и при $q=0$ верните вход без изменения. В режиме обучения выберите
независимую маску для каждого элемента и сохраните масштаб в среднем. Для $q$ вне
интервала $[0,1)$ выбросьте `ValueError`. Сам вход изменять нельзя.

<details><summary>Подсказка</summary>
`torch.rand_like(x)` создаёт равномерные числа на том же устройстве и той же формы.
Сравнение с $q$ даёт булеву маску. Не создавайте одну маску на весь батч и не отключайте
параметры сети навсегда: случайность относится к текущему прямому проходу.
</details>


In [ ]:
def inverted_dropout(x, q=0.5, training=True):
    # TODO E9: проверьте q, обработайте eval/q=0, затем примените маску и масштабирование.
    raise NotImplementedError('E9: реализуйте inverted dropout')


In [ ]:
seed_everything(12)
dx = torch.ones(40000, requires_grad=True)
dy = inverted_dropout(dx, q=0.5, training=True)
assert dy.shape == dx.shape and dy.dtype == dx.dtype and dy.device == dx.device
assert set(dy.detach().unique().tolist()) == {0.0, 2.0}, 'Проверьте масштаб сохранённых элементов.'
assert abs(dy.mean().item() - 1.0) < 0.03
torch.testing.assert_close(dx.detach(), torch.ones_like(dx))
dy.sum().backward()
torch.testing.assert_close(dx.grad, dy.detach())
arbitrary_x = torch.tensor([[2., -3., 4.]])
torch.testing.assert_close(inverted_dropout(arbitrary_x, 0.7, False), arbitrary_x)
torch.testing.assert_close(inverted_dropout(arbitrary_x, 0.0, True), arbitrary_x)
for bad_q in [-0.1, 1.0]:
    try:
        inverted_dropout(arbitrary_x, bad_q, True)
    except ValueError:
        pass
    else:
        raise AssertionError('Не проверяется допустимый диапазон q.')
print('E9 ✓ — маска, ожидание, градиент и режим оценки проверены')


In [ ]:
seed_everything(SEED)
activation_example = torch.tensor([1., 2., 3., 4., 5., 6.])
print('Активации: ', activation_example.tolist())
for _ in range(3):
    print('Обучение:  ', inverted_dropout(activation_example, q=0.5, training=True).tolist())
print('Оценка:    ', inverted_dropout(activation_example, q=0.5, training=False).tolist())


### Сравниваем одну архитектуру с dropout и без него

Вставим вашу функцию после каждой ReLU. Ни число обучаемых параметров, ни начальные
веса не меняются. Сравним с MLP из раздела 5 при тех же данных, батче, learning rate
и числе эпох. Train-метрики обеих моделей оцениваются с выключенным dropout.

**Прогноз до запуска:** как dropout повлияет на train loss? Обязательно ли он улучшит
validation за то же число эпох? …


In [ ]:
class MyDropout(nn.Module):
    def __init__(self, q):
        super().__init__()
        self.q = q

    def forward(self, x):
        return inverted_dropout(x, self.q, self.training)

def make_dropout_mlp(q, state):
    base = make_mlp()
    base.load_state_dict(state)
    layers = []
    for layer in base.children():
        layers.append(layer)
        if isinstance(layer, nn.ReLU):
            layers.append(MyDropout(q))
    return nn.Sequential(*layers)


In [ ]:
DROPOUT_Q = 0.5
dropout_model = make_dropout_mlp(DROPOUT_Q, initial_state)
# При выключенной маске исходные сети должны совпадать.
same_start = make_mlp()
same_start.load_state_dict(initial_state)
dropout_model.eval()
with torch.no_grad():
    comparison_x = train_ds.tensors[0][:16]
    torch.testing.assert_close(dropout_model(comparison_x), same_start(comparison_x))

dropout_run = fit(dropout_model, batch_size=BASE_BATCH, epochs=EPOCHS, lr=LEARNING_RATE)
dropout_results = {'MLP без dropout': mlp_run, 'MLP с dropout': dropout_run}
plot_learning(dropout_results)
print_results(dropout_results)


**Запишите результат.**

1. Изменились ли train loss, validation loss и accuracy в одном направлении? …
2. Если dropout хуже, какие объяснения, кроме «метод не работает», стоит проверить? …
3. Почему для сравнения train-метрик мы отключили dropout? …

Один запуск не устанавливает преимущество метода. Dropout меняет задачу обучения,
может замедлять подгонку и требует выбора своей силы и режима оптимизации.
Если он не улучшил качество за 10 эпох, это ограниченный результат именно нашего пилота.
На следующий день можно менять объём данных, $q$, длительность обучения и повторять
сравнения с несколькими seed.

**Связь со статьёй Srivastava.** Там случайное отключение нейронов используется против
переобучения и интерпретируется через подсети с общими весами. Сегодня воспроизведена
механика и поставлено маленькое сравнение, а не воспроизведены все результаты работы.
[Оригинальная статья](https://jmlr.org/papers/volume15/srivastava14a/srivastava14a.pdf).


## 7. Что берём с собой на второй день

Сформулируйте ответы своими словами, без запуска дополнительных ячеек:

1. Что является данными, параметрами и гиперпараметрами в нашем эксперименте?
2. Почему несколько Linear без нелинейности не дают принципиально более сложную функцию?
3. Чем отличаются backward и шаг оптимизатора?
4. Почему accuracy и cross-entropy могут изменяться по-разному?
5. Что меняется вместе с батчем при фиксированном числе эпох?
6. Где появляется случайность в mini-batch обучении, а где — в dropout?
7. Почему один успешный запуск ещё не доказывает вывод статьи?

**Мини-отчёт пары, 5 предложений:**

- Мы проверяли гипотезу: …
- Изменяли: …; фиксировали: …
- На графике наблюдаем: …
- Пока нельзя заключить: …
- Следующий контрольный эксперимент: …

**Задание по желанию.** Выберите только одну дополнительную настройку: размер батча,
learning rate или q. До запуска запишите прогноз, измените только эту настройку,
заново обучите модель с исходных весов и объясните результат. Не выбирайте настройки по test.

Тема будущего исследования может быть любой из двух. Для батчей естественное продолжение —
число шагов и геометрия loss. Для dropout — объём данных, сила регуляризации и способы
усреднения предсказаний. Общая инфраструктура обучения уже готова.


## Если что-то пошло не так

| Симптом | Что проверить |
|---|---|
| `NotImplementedError: E...` | Это пропуск задания. Замените заглушку и повторите определение функции. |
| Loss около 2.30 и почти не меняется | Есть ли step, подходящий ли learning rate, не перепутаны ли метки? |
| Ошибка размерностей | Вход `[B,1,28,28]`, после Flatten `[B,784]`, выход `[B,10]`. |
| Loss быстро становится `nan` | Слишком большой шаг, отсутствие очистки градиентов, нестабильное вычисление экспонент. |
| Ошибка типа меток | Нужны целые индексы классов `torch.long`, не вещественные числа. |
| Тензоры на разных устройствах | И модель, и x, и y должны быть на одном device. |
| Изменили функцию, а поведение старое | Снова выполните ячейку с определением; затем зависимые ячейки. |
| Изменили EPOCHS, но график прежний | Переобучите все сравниваемые модели с одинаковым новым бюджетом. |
| GPU недоступен | Выберите CPU; при нехватке времени уменьшайте EPOCHS одинаково во всех сравнениях. |

При изменении архитектуры заново выполните раздел 5 и оба нужных продолжения:
сохранённое `initial_state` должно соответствовать новой архитектуре.


## Материалы для чтения после занятия

- [PyTorch: основы и цикл обучения](https://docs.pytorch.org/tutorials/beginner/basics/quickstart_tutorial.html).
- [Autograd и вычислительный граф](https://docs.pytorch.org/tutorials/beginner/basics/autogradqs_tutorial.html).
- [CrossEntropyLoss: входы и усреднение](https://docs.pytorch.org/docs/stable/generated/torch.nn.CrossEntropyLoss.html).
- [MNIST в torchvision](https://docs.pytorch.org/vision/stable/generated/torchvision.datasets.MNIST.html).
- [Dropout в PyTorch](https://docs.pytorch.org/docs/stable/generated/torch.nn.Dropout.html).
- [Keskar et al., 2017](https://arxiv.org/abs/1609.04836): abstract, разделы 1–2 и Figure 3.
- [Srivastava et al., 2014](https://jmlr.org/papers/v15/srivastava14a.html): введение, разделы 4–5 и 7.

Формулы и упражнения этого ноутбука служат введением в DL. Маленькие модели и бюджеты
выбраны для занятия; они отличаются от полных экспериментальных протоколов статей.
